# uMedSum baseline on the agentic SOAP-note consultations (RQ1-matched)

Runs the full uMedSum pipeline (the baseline reproduced in `Baseline_Implementation`) on the **110 simulated consultations** in `agentic-pipeline-output/soap_notes`, set up exactly like Verify-and-Ask:

| | uMedSum (this notebook) | Verify-and-Ask (ours) |
|---|---|---|
| Input | `consultation.md` only | `consultation.md` only |
| Stage 1 | Element-Aware + 1-shot ICL, full SOAP (LLaMA-3-8B) | strict grounded full SOAP (LLaMA-3-8B) |
| A/P | generated in Stage 1, **passed through unchanged** | generated in Stage 1, **passed through unchanged** |
| Stage 2 | RTB-TS on **S/O only** (DeBERTa NLI, Te 0.9 / Tc 0.8 / Ta 0.5) | sentence-first NLI verification on S/O |
| Stage 3 | MMR + KeyBERT coverage (cov_min 0.4, top_m 2) + GPT-2 insertion on **S/O only** | omission recovery + Ask on S/O |
| Reference | agentic `soap_note.md` S/O (evaluation only) | same |
| Split | `prepared_dataset.json`, seed 42, 22 dev / 88 test | same |
| Scorer | `Stage23_pipeline/code/evaluate.py` | same |

**ICL leakage:** the 1-shot demo is D2N002, a **dev** case. The runner stops if a demo is ever a test case. Headline numbers = **test (88)**.

**Stage 2 header fix (on by default):** the baseline's `clean_summary_text()` drops *any* line containing EXAM / PLAN / HISTORY OF PRESENT ILLNESS / CHIEF COMPLAINT / ... before NLI scoring. That is about 1/3 of content lines in the earlier 140-case baseline run. This runner only skips real heading lines. Pass `--reproduce-header-bug` to run the baseline exactly as before.

Everything is written to Drive and resumes per case if Colab disconnects. **Runtime: GPU (A100 or L4 recommended; T4 works but is slower).**

## 1. Mount Drive and check paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
ROOT = "/content/drive/MyDrive/tokenizers"
PIPE = f"{ROOT}/Baseline_Implementation/umedsum_soap_notes"
OUT  = f"{PIPE}/output"
RQ1_CODE = f"{ROOT}/verify_and_ask_pipeline/Stage23_pipeline/code"
RQ1_EVAL = f"{ROOT}/verify_and_ask_pipeline/Stage23_pipeline/output/evaluation"
DATASET  = f"{ROOT}/verify_and_ask_pipeline/stage1_pipeline/stage1_output/prepared_dataset.json"

need = {
    "prepared dataset (110 cases, 22/88)": DATASET,
    "agentic soap notes": f"{ROOT}/agentic-pipeline-output/soap_notes",
    "baseline Stage 1-3 code": f"{ROOT}/Baseline_Implementation/umedsum_pipeline_baseline/stage_2/confabulation_remover.py",
    "ICL examples": f"{ROOT}/Baseline_Implementation/aci_dataset_rq1_baseline/icl_examples.json",
    "RQ1 evaluate.py": f"{RQ1_CODE}/evaluate.py",
    "RQ1 reference facts": f"{RQ1_EVAL}/reference_facts.json",
    "RQ1 test results": f"{RQ1_EVAL}/results_test.json",
}
for name, p in need.items():
    print(("OK      " if os.path.exists(p) else "MISSING ") + name + " -> " + p)

d = json.load(open(DATASET))
from collections import Counter
print("\nCases:", len(d["records"]), Counter(r["split"] for r in d["records"]), "seed", d["split_seed"])
os.makedirs(PIPE, exist_ok=True)

Mounted at /content/drive
OK      prepared dataset (110 cases, 22/88) -> /content/drive/MyDrive/tokenizers/verify_and_ask_pipeline/stage1_pipeline/stage1_output/prepared_dataset.json
OK      agentic soap notes -> /content/drive/MyDrive/tokenizers/agentic-pipeline-output/soap_notes
OK      baseline Stage 1-3 code -> /content/drive/MyDrive/tokenizers/Baseline_Implementation/umedsum_pipeline_baseline/stage_2/confabulation_remover.py
OK      ICL examples -> /content/drive/MyDrive/tokenizers/Baseline_Implementation/aci_dataset_rq1_baseline/icl_examples.json
OK      RQ1 evaluate.py -> /content/drive/MyDrive/tokenizers/verify_and_ask_pipeline/Stage23_pipeline/code/evaluate.py
OK      RQ1 reference facts -> /content/drive/MyDrive/tokenizers/verify_and_ask_pipeline/Stage23_pipeline/output/evaluation/reference_facts.json
OK      RQ1 test results -> /content/drive/MyDrive/tokenizers/verify_and_ask_pipeline/Stage23_pipeline/output/evaluation/results_test.json

Cases: 110 Counter({'test': 88, 'dev'

## 2. Write the pipeline scripts into Drive
`run_umedsum_soap.py` imports the unchanged Stage 1/2/3 modules from `umedsum_pipeline_baseline/` and adds only the S/O vs A/P handling.

In [2]:
%%writefile /content/drive/MyDrive/tokenizers/Baseline_Implementation/umedsum_soap_notes/run_umedsum_soap.py
"""
uMedSum (Nagar et al., ACL 2025) on the 110 simulated consultations, matched to the RQ1 setup.

This is baseline B3 from the proposal ("Aegle intake -> note -> uMedSum filter"), run end to end:

  Stage 1  Element-Aware + 1-shot ICL full-SOAP generation (LLaMA-3-8B)   <- same prompts as the baseline
           -> split into S/O and A/P. A/P is kept aside and never changed.
  Stage 2  RTB-TS confabulation removal (DeBERTa-v3-large NLI, Te=0.9, Tc=0.8, Ta=0.5)   on S/O only
  Stage 3  MMR key sentences + KeyBERT coverage (cov_min=0.4, top_m=2) + GPT-2 perplexity insertion   on S/O only
  Final    Stage 3 S/O + the untouched Stage 1 A/P

Input is the same prepared dataset Verify-and-Ask uses (built from agentic-pipeline-output/soap_notes,
seed 42, 22 dev / 88 test). Only the consultation is given to the models; the agentic soap_note.md S/O is
carried in each record solely for evaluation.

Output records use the Verify-and-Ask record format (stage1_so / stage2_so / stage3_so, stage1_ap,
ap_unchanged, stage3_added, ...), so verify_and_ask_pipeline/Stage23_pipeline/code/evaluate.py scores
both systems with exactly the same code. Systems A/B/C in that evaluation = uMedSum Stage 1/2/3.

The Stage 1/2/3 algorithms are imported unchanged from Baseline_Implementation/umedsum_pipeline_baseline.
The only adaptation is section handling (S/O vs A/P), plus an optional fix to the baseline's header
detection (see FixedHeaderRemover), on by default.

  python run_umedsum_soap.py --limit 2            # smoke test
  python run_umedsum_soap.py                      # all 110 cases (resumes per case)
  python run_umedsum_soap.py --split test         # only the 88 test cases
"""
import argparse
import json
import re
import sys
import time
import traceback
from pathlib import Path

HERE = Path(__file__).resolve().parent
REPO = HERE.parent.parent  # .../tokenizers
BASELINE = REPO / "Baseline_Implementation" / "umedsum_pipeline_baseline"
for sub in ("stage_1", "stage_2", "stage_3"):
    sys.path.insert(0, str(BASELINE / sub))

DEFAULT_DATASET = REPO / "verify_and_ask_pipeline" / "stage1_pipeline" / "stage1_output" / "prepared_dataset.json"
DEFAULT_ICL = REPO / "Baseline_Implementation" / "aci_dataset_rq1_baseline" / "icl_examples.json"
DEFAULT_OUT = HERE / "output"


# ---------------------------------------------------------------------------------------------
# Section handling
# ---------------------------------------------------------------------------------------------
_MAIN_HEADING = re.compile(
    r"^(?P<name>subjective|objective|assessment\s*(?:and|&|/)\s*plan|assessment|plan|a\s*/\s*p)\b"
    r"(?:\s*\([^)]*\))?\s*(?P<colon>:)?\s*(?P<rest>.*)$",
    re.IGNORECASE,
)
# Used only if the model wrote no SUBJECTIVE / OBJECTIVE / ASSESSMENT headings at all.
_FALLBACK_SUBHEADINGS = [
    ("subjective", r"chief complaint|history of present illness|hpi|review of systems|past medical history|"
                   r"medications?|allergies|social history|family history|cc"),
    ("objective", r"physical exam(?:ination)?|exam(?:ination)?|vitals?|vital signs|diagnostic|results?|labs?|"
                  r"imaging|investigations?"),
    ("ap", r"diagnos[ie]s|impression|clinical reasoning|treatment|follow[- ]?up|recommendations?"),
]


def _strip_markup(line: str) -> str:
    s = line.strip()
    s = re.sub(r"^#{1,6}\s*", "", s)
    s = re.sub(r"^(?:[-*+•]|\d+[.)]|[IVX]+[.)])\s+", "", s)
    s = s.replace("**", "").replace("__", "").strip()
    return s


def _main_heading(line: str):
    """Return (section, rest_of_line) if the line opens a SOAP section, else None."""
    s = _strip_markup(line)
    m = _MAIN_HEADING.match(s)
    if not m:
        return None
    rest = m.group("rest").strip()
    # "Subjective findings include ..." is prose, not a heading: need a colon or nothing after the name.
    if rest and not m.group("colon"):
        return None
    name = m.group("name").lower().replace(" ", "")
    section = "subjective" if name.startswith("subj") else "objective" if name.startswith("obj") else "ap"
    return section, rest.strip("* ")


def _fallback_heading(line: str):
    s = _strip_markup(line)
    head, _, rest = s.partition(":")
    head = head.strip()
    if not head or len(head.split()) > 6:
        return None
    if not (s.endswith(":") or rest):  # needs "Heading:" or "Heading: text"
        return None
    for section, pat in _FALLBACK_SUBHEADINGS:
        if re.fullmatch(rf"(?:{pat})(?:\s*\(.*\))?(?:\s*(?:&|and)\s*.*)?", head, re.IGNORECASE):
            return section, rest.strip()
    return None


_PREAMBLE = re.compile(r"^(here is|here's|below is|this is the|soap note|sure[,!.]|certainly)", re.IGNORECASE)


def split_umedsum_note(note: str) -> dict:
    """Split a Stage 1 uMedSum note into S, O and A/P.

    The baseline prompt asks for 'SUBJECTIVE:', 'OBJECTIVE:', 'ASSESSMENT AND PLAN:'. Text before the
    first heading (chat preambles) is dropped. If no main headings are found, the baseline's own
    sub-headings (Chief Complaint, Physical Examination, ...) are used instead.
    """
    out = {"subjective": [], "objective": [], "ap": [], "preamble": []}
    lines = note.splitlines()
    found = {"subjective": False, "objective": False, "ap": False}
    for detector in (_main_heading, _fallback_heading):
        out = {"subjective": [], "objective": [], "ap": [], "preamble": []}
        found = {"subjective": False, "objective": False, "ap": False}
        current = "preamble"
        for line in lines:
            h = detector(line)
            if h:
                current, rest = h
                found[current] = True
                if rest:
                    out[current].append(rest)
                continue
            out[current].append(line.rstrip())
        if found["subjective"] or found["objective"]:
            mode = "main_headings" if detector is _main_heading else "subheadings"
            break
    else:
        # No recognisable headings: keep the whole note as S/O so nothing is silently lost (flagged).
        mode = "unparsed"
        out = {"subjective": lines, "objective": [], "ap": [], "preamble": []}
    clean = lambda xs: "\n".join(xs).strip()
    return {
        "subjective": clean(out["subjective"]),
        "objective": clean(out["objective"]),
        "assessment_and_plan": clean(out["ap"]),
        "dropped_preamble": clean(out["preamble"]),
        "parse_mode": mode,
        "parse_ok": mode != "unparsed" and found["ap"],
    }


# ---------------------------------------------------------------------------------------------
# Stage 2 helper: the baseline's unit splitter, with the header bug fixed
# ---------------------------------------------------------------------------------------------
def make_remover_class(fix_header_bug: bool):
    from confabulation_remover import ConfabulationRemover

    class SectionRemover(ConfabulationRemover):
        """Runs RTB-TS on one section at a time.

        The baseline's clean_summary_text() treats ANY line that contains 'EXAM', 'PLAN', 'OBJECTIVE',
        'ASSESSMENT', 'SUBJECTIVE', 'CHIEF COMPLAINT' or 'HISTORY OF PRESENT ILLNESS' as a heading and
        drops it before NLI scoring (e.g. 'Physical examination revealed ...', 'He explained ...').
        Its bullet regex also strips leading numbers ('50-year-old' -> '-year-old'). With
        fix_header_bug=True only short heading-like lines are skipped and only real bullets are
        stripped; scoring and thresholds are unchanged. Set False to reproduce the baseline exactly.
        """
        forced_section = "SUBJECTIVE"

        def clean_summary_text(self, text):
            if not fix_header_bug:
                units = super().clean_summary_text(text)
                for u in units:
                    u["section"] = self.forced_section
                return units
            units = []
            for line in text.splitlines():
                s = line.strip()
                if not s:
                    continue
                hdr = _strip_markup(s).rstrip(":").strip()
                if (":" not in hdr and len(hdr.split()) <= 6 and not re.search(r"[.!?]$", hdr) and (
                        s.endswith(":") or _main_heading(hdr) or _fallback_heading(hdr + ":"))):
                    continue  # a bare heading such as 'Chief Complaint:' or 'Physical Examination & Vitals'
                if re.match(r"^(here is|here's|below is|this is the|soap note:?)", s, re.IGNORECASE):
                    continue
                if re.match(r"^(please note|note:|disclaimer:)", s, re.IGNORECASE):
                    continue
                body = re.sub(r"^(?:[-*+•]|\d+[.)])\s+", "", s).replace("**", "").replace("__", "").strip()
                if len(body.split()) < 3:
                    continue
                for seg in self.segmenter.segment(body):
                    seg = seg.strip()
                    if len(seg.split()) >= 3:
                        units.append({"unit": seg, "section": self.forced_section})
            return units

    return SectionRemover


# ---------------------------------------------------------------------------------------------
# Stage 3 helper: Algorithm 2 (GPT-2 perplexity insertion) over S/O with section tracking
# ---------------------------------------------------------------------------------------------
def perplexity_insert(merger, units, missing):
    """units: list of (section, sentence). Each missing sentence goes where total perplexity is lowest
    (baseline Algorithm 2). Its section is that of the preceding sentence (or the next one at index 0)."""
    units = list(units)
    log = []
    for sent in missing:
        if not units:
            units.append(("subjective", sent))
            log.append({"inserted_sentence": sent, "location_index": 0, "section": "subjective", "best_ppl": None})
            continue
        best_loc, best_ppl = 0, float("inf")
        texts = [u[1] for u in units]
        for loc in range(len(texts) + 1):
            ppl = merger.calculate_perplexity(" ".join(texts[:loc] + [sent] + texts[loc:]))
            if ppl < best_ppl:
                best_loc, best_ppl = loc, ppl
        section = units[best_loc - 1][0] if best_loc > 0 else units[0][0]
        units.insert(best_loc, (section, sent))
        log.append({"inserted_sentence": sent, "location_index": best_loc, "section": section,
                    "best_ppl": round(best_ppl, 2)})
    return units, log


def units_to_so(units):
    return {"subjective": "\n".join(t for s, t in units if s == "subjective"),
            "objective": "\n".join(t for s, t in units if s == "objective")}


# ---------------------------------------------------------------------------------------------
# Per-case pipeline
# ---------------------------------------------------------------------------------------------
def run_case(rec, ctx, args):
    from stage1_generator import run_stage1_element_aware

    t0 = time.time()
    conv = rec["conversation"]
    out = {"case_id": rec["case_id"], "split": rec["split"], "is_icl_demo": rec["case_id"] in ctx["icl_ids"]}

    # ---- Stage 1: unchanged baseline generator (Element-Aware 2-step + 1-shot ICL) ----------------
    s1 = run_stage1_element_aware(ctx["client"], {"src": conv}, icl_prefix=ctx["icl_prefix"])
    parsed = split_umedsum_note(s1["stage1_summary"])
    stage1_so = {"subjective": parsed["subjective"], "objective": parsed["objective"]}
    stage1_ap = {"assessment_and_plan": parsed["assessment_and_plan"]}
    out.update({
        "stage1_elements": s1["stage1_elements"],
        "stage1_raw_note": s1["stage1_summary"],
        "stage1_parse": {k: parsed[k] for k in ("parse_mode", "parse_ok", "dropped_preamble")},
        "stage1_so": stage1_so,
        "stage1_ap": stage1_ap,
    })

    # ---- Stage 2: RTB-TS on S and O separately (A/P never enters) ---------------------------------
    remover = ctx["remover"]
    stage2_units, details, removed, n_claims, n_kept = [], {}, [], 0, 0
    for section in ("subjective", "objective"):
        text = stage1_so[section]
        if not text.strip():
            details[section] = []
            continue
        remover.forced_section = section.upper()
        res = remover.process_summary(conv, text)
        stage2_units += [(section, u) for u in res["faithful_units"]]
        details[section] = res["details"]
        for d in res["details"]:
            n_claims += 1
            kept = d["classification"] == "faithful" or bool(d.get("atomic_decomposition", {}).get("retained"))
            n_kept += int(kept)
            if d["classification"] == "confabulated":
                removed.append({"section": section, "claim": d["unit"], "scores": d["scores"], "reason": d["reasoning"]})
            for a in d.get("atomic_decomposition", {}).get("discarded", []):
                removed.append({"section": section, "claim": a["atom"], "entailment": a["entailment"],
                                "reason": a["reasoning"], "from_unit": d["unit"]})
    stage2_so = units_to_so(stage2_units)
    counts = {}
    for sec in details.values():
        for d in sec:
            counts[d["classification"]] = counts.get(d["classification"], 0) + 1
    out.update({
        "stage2_so": stage2_so,
        "stage2_audit": details,
        "stage2_stats": {"decision_counts": counts, "num_claims": n_claims, "num_kept": n_kept,
                         "claim_retention_rate": round(n_kept / n_claims, 4) if n_claims else None},
    })

    # ---- Stage 3: missing-information addition on S/O --------------------------------------------
    extractor, detector, merger = ctx["extractor"], ctx["detector"], ctx["merger"]
    key_sents = extractor.extract_document_key_sentences(conv, top_m=args.top_m)
    so_flat = " ".join(t for _, t in stage2_units)
    phrases = extractor.extract_summary_key_phrases(so_flat, top_n=10)
    detection = detector.detect_missing_sentences(key_sents, phrases)
    missing = detection["missing_sentences"]
    if args.no_gpt2:
        final_units = stage2_units + [("subjective", m) for m in missing]
        ins_log = [{"inserted_sentence": m, "location_index": None, "section": "subjective", "best_ppl": None}
                   for m in missing]
    else:
        final_units, ins_log = perplexity_insert(merger, stage2_units, missing)
    stage3_so = units_to_so(final_units)
    cov = {c["sentence"]: c["score"] for c in detection["coverage_scores"]}
    out.update({
        "stage3_so": stage3_so,
        "stage3_key_sentences": key_sents,
        "stage3_summary_key_phrases": phrases,
        "stage3_coverage_scores": detection["coverage_scores"],
        # Same shape as Verify-and-Ask so evaluate.py computes gap-detection / addition metrics.
        "stage3_candidate_log": [{"candidate": m, "statement": m, "coverage": cov.get(m), "status": "added",
                                  "reason": "coverage <= cov_min (uMedSum adds the dialogue sentence verbatim)",
                                  "insertion": {"section": l["section"], "line_index": l["location_index"],
                                                "perplexity": l["best_ppl"]}}
                                 for m, l in zip(missing, ins_log)],
        "stage3_added": [{"statement": m, "evidence": m, "section": l["section"], "candidate": m}
                         for m, l in zip(missing, ins_log)],
        "stage3_stats": {"num_key_sentences": len(key_sents), "num_flagged": len(missing), "num_added": len(missing)},
        "gaps": {"gaps": [], "dropped": [], "num_generated": 0},  # uMedSum has no Ask stage
        "unsupported_claims": {"removed_stage2": removed},
    })

    # ---- Final: Stage 3 S/O + untouched Stage 1 A/P ----------------------------------------------
    final_ap = dict(stage1_ap)
    out.update({
        "final_ap": final_ap,
        "ap_unchanged": final_ap == stage1_ap and final_ap["assessment_and_plan"] == parsed["assessment_and_plan"],
        "thresholds": {"Te": args.te, "Tc": args.tc, "Ta": args.ta, "cov_min": args.cov_min, "top_m": args.top_m,
                       "fix_header_bug": not args.reproduce_header_bug},
        "models": {"llm": args.model, "nli": args.nli_model, "embedder": "all-MiniLM-L6-v2", "ppl": args.ppl_model},
        "status": "success",
        "seconds": round(time.time() - t0, 1),
        "conversation": conv,
        "reference_so": rec["reference_so"],  # evaluation only; never passed to a model
    })
    return out


def write_final_note(path, r):
    so, ap = r["stage3_so"], r["final_ap"]["assessment_and_plan"]
    path.write_text(
        f"# {r['case_id']} - uMedSum final note\n\n## Subjective\n{so['subjective'] or '(empty)'}\n\n"
        f"## Objective\n{so['objective'] or '(empty)'}\n\n## Assessment and Plan (Stage 1, unchanged)\n{ap or '(empty)'}\n",
        encoding="utf-8")


def main():
    ap = argparse.ArgumentParser(description="uMedSum on the agentic SOAP-note consultations (RQ1-matched)")
    ap.add_argument("--dataset", default=str(DEFAULT_DATASET))
    ap.add_argument("--icl-json", default=str(DEFAULT_ICL))
    ap.add_argument("--n-shots", type=int, default=1)
    ap.add_argument("--out-dir", default=str(DEFAULT_OUT))
    ap.add_argument("--split", choices=["dev", "test", "all"], default="all")
    ap.add_argument("--limit", type=int, default=None)
    ap.add_argument("--model", default="llama3:8b")
    ap.add_argument("--ollama-url", default="http://localhost:11434")
    ap.add_argument("--nli-model", default="MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli")
    ap.add_argument("--ppl-model", default="gpt2")
    ap.add_argument("--te", type=float, default=0.9)
    ap.add_argument("--tc", type=float, default=0.8)
    ap.add_argument("--ta", type=float, default=0.5)
    ap.add_argument("--cov-min", type=float, default=0.4)
    ap.add_argument("--top-m", type=int, default=2)
    ap.add_argument("--no-gpt2", action="store_true", help="append missing sentences instead of GPT-2 placement")
    ap.add_argument("--reproduce-header-bug", action="store_true",
                    help="use the baseline's original line filter in Stage 2 (drops lines containing EXAM/PLAN/...)")
    ap.add_argument("--retry-errors", action="store_true")
    ap.add_argument("--overwrite", action="store_true")
    args = ap.parse_args()

    out_dir = Path(args.out_dir)
    cases_dir, notes_dir = out_dir / "cases", out_dir / "final_notes"
    cases_dir.mkdir(parents=True, exist_ok=True)
    notes_dir.mkdir(parents=True, exist_ok=True)

    data = json.loads(Path(args.dataset).read_text(encoding="utf-8"))
    records = data["records"]
    n_dev = sum(r["split"] == "dev" for r in records)
    n_test = sum(r["split"] == "test" for r in records)
    print(f"Dataset: {len(records)} cases ({n_dev} dev / {n_test} test), seed {data.get('split_seed')}")
    if (len(records), n_dev, n_test) != (110, 22, 88):
        print("[WARN] expected 110 cases (22 dev / 88 test) - check you are using the RQ1 prepared_dataset.json")

    # ---- leakage guard: the ICL demonstration must not be a test case ----------------------------
    icl_data = json.loads(Path(args.icl_json).read_text(encoding="utf-8"))["data"][: args.n_shots]
    icl_ids = {d.get("encounter_id") or d.get("file", "").split("-")[0] for d in icl_data}
    test_ids = {r["case_id"] for r in records if r["split"] == "test"}
    if icl_ids & test_ids:
        sys.exit(f"[LEAKAGE] ICL demonstration(s) {sorted(icl_ids & test_ids)} are test cases. "
                 f"Use --n-shots 1 (D2N002, a dev case) or change icl_examples.json.")
    print(f"ICL demonstration(s): {sorted(icl_ids)} - none in the test split. "
          f"Dev cases that are also demos: {sorted(icl_ids & {r['case_id'] for r in records})}")

    if args.split != "all":
        records = [r for r in records if r["split"] == args.split]
    if args.limit:
        records = records[: args.limit]

    # ---- models -----------------------------------------------------------------------------------
    from ollama_client import OllamaClient
    from prompts import format_icl_demonstrations
    from nli_scorer import DebertaNLIScorer
    from atomic_decomposer import AtomicDecomposer
    from key_extractor import KeyExtractor
    from missing_detector import MissingInformationDetector
    from perplexity_merger import PerplexityMerger

    client = OllamaClient(base_url=args.ollama_url, model=args.model)
    if not client.check_connection():
        sys.exit(f"Ollama is not reachable at {args.ollama_url}. Start `ollama serve` and pull {args.model}.")
    if not any(args.model in m for m in client.list_models()):
        sys.exit(f"Model {args.model} is not pulled. Run: ollama pull {args.model}")
    Remover = make_remover_class(fix_header_bug=not args.reproduce_header_bug)
    extractor = KeyExtractor()
    ctx = {
        "client": client,
        "icl_prefix": format_icl_demonstrations(icl_data, n_shots=args.n_shots),
        "icl_ids": icl_ids,
        "remover": Remover(DebertaNLIScorer(model_name=args.nli_model),
                           AtomicDecomposer(ollama_url=args.ollama_url, model=args.model),
                           te=args.te, tc=args.tc, ta=args.ta),
        "extractor": extractor,
        "detector": MissingInformationDetector(embedder=extractor.embedder, cov_min=args.cov_min),
        "merger": None if args.no_gpt2 else PerplexityMerger(model_name=args.ppl_model),
    }

    # ---- run, one checkpoint file per case ---------------------------------------------------------
    t_start = time.time()
    for i, rec in enumerate(records, 1):
        cpath = cases_dir / f"{rec['case_id']}.json"
        if cpath.exists() and not args.overwrite:
            prev = json.loads(cpath.read_text(encoding="utf-8"))
            if prev.get("status") == "success" or not args.retry_errors:
                print(f"[{i}/{len(records)}] {rec['case_id']}: cached ({prev.get('status')})")
                continue
        print(f"[{i}/{len(records)}] {rec['case_id']} ({rec['split']}) ...", end="", flush=True)
        try:
            r = run_case(rec, ctx, args)
            write_final_note(notes_dir / f"{rec['case_id']}.md", r)
            st = r["stage2_stats"]
            print(f" done {r['seconds']}s | parse={r['stage1_parse']['parse_mode']} | "
                  f"S2 kept {st['num_kept']}/{st['num_claims']} | S3 added {r['stage3_stats']['num_added']}")
        except Exception as exc:
            r = {"case_id": rec["case_id"], "split": rec["split"], "status": "error",
                 "error": f"{type(exc).__name__}: {exc}", "traceback": traceback.format_exc()}
            print(f" FAILED: {r['error']}")
        cpath.write_text(json.dumps(r, ensure_ascii=False, indent=2), encoding="utf-8")

    # ---- combine every case checkpoint into one file (the evaluator's input) ----------------------
    all_recs = [json.loads(p.read_text(encoding="utf-8")) for p in sorted(cases_dir.glob("*.json"))]
    ok = [r for r in all_recs if r.get("status") == "success"]
    combined = {"system": "uMedSum (Element-Aware+1-shot ICL -> RTB-TS -> MMR/GPT-2), S/O only, A/P passthrough",
                "num_cases": len(all_recs), "num_success": len(ok), "num_errors": len(all_recs) - len(ok),
                "icl_demo_ids": sorted(icl_ids), "records": all_recs}
    (out_dir / "umedsum_output.json").write_text(json.dumps(combined, ensure_ascii=False, indent=2), encoding="utf-8")

    parse_modes = {}
    for r in ok:
        m = r["stage1_parse"]["parse_mode"] + ("" if r["stage1_parse"]["parse_ok"] else " (no A/P heading)")
        parse_modes[m] = parse_modes.get(m, 0) + 1
    print(f"\nFinished in {round(time.time() - t_start)}s. {len(ok)}/{len(all_recs)} cases succeeded.")
    print("Stage 1 section parsing:", parse_modes)
    print("A/P unchanged rate:", round(sum(r["ap_unchanged"] for r in ok) / len(ok), 4) if ok else None)
    print("Saved", out_dir / "umedsum_output.json")


if __name__ == "__main__":
    main()

Writing /content/drive/MyDrive/tokenizers/Baseline_Implementation/umedsum_soap_notes/run_umedsum_soap.py


In [3]:
%%writefile /content/drive/MyDrive/tokenizers/Baseline_Implementation/umedsum_soap_notes/compare_results.py
"""
Side-by-side comparison: uMedSum (baseline B3) vs Verify-and-Ask (ours) on the same split, same metrics.

Reads what evaluate.py already produced for each system (results_<split>.json plus its per-case caches),
so no model is re-run. Writes:
  comparison_<split>.md / .json    aggregate table (every metric, all stages of both systems)
  paired_tests_<split>.csv         per-metric paired test on the final notes (case by case)

Paired tests use the cases both systems completed. For each metric: mean of each system, mean difference
(ours - uMedSum), 95% bootstrap CI of the difference (10k resamples, seed 42) and the Wilcoxon
signed-rank p-value.

  python compare_results.py --split test
"""
import argparse
import csv
import json
import random
from pathlib import Path
from statistics import mean

HERE = Path(__file__).resolve().parent
REPO = HERE.parent.parent
NLI_TAG = "DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
TE, TA, TCONTRA = 0.90, 0.50, 0.50  # evaluate.py defaults (config.py)
STAGES = {"A_stage1": "stage1_so", "B_verified": "stage2_so", "C_final": "stage3_so"}

# Direction: +1 higher is better, -1 lower is better, 0 informational.
DIRECTION = {
    "unsupported_claim_rate": -1, "contradiction_rate": -1, "uncertain_rate": -1,
    "unsupported_addition_rate": -1, "duplicate_addition_rate": -1, "question_redundancy_rate": -1,
    "num_claims": 0, "num_genuine_omissions": 0, "num_added": 0, "questions_kept": 0,
}


def load(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))


def so_text(so):
    return "\n".join(x for x in ((so.get("subjective") or "").strip(), (so.get("objective") or "").strip()) if x)


# ------------------------------------------------------------------ per-case metrics
def per_case(records, eval_dir, system_key):
    """case_id -> {metric: value} for one stage of one system, from records + evaluate.py caches."""
    from rouge_score import rouge_scorer
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeLsum"], use_stemmer=True)
    stage = STAGES[system_key]
    g = eval_dir / f"grounding_cache_{NLI_TAG}.json"
    f = eval_dir / f"fact_scores_cache_{NLI_TAG}.json"
    facts_p = eval_dir / "reference_facts.json"
    gcache = load(g) if g.exists() else {}
    fcache = load(f) if f.exists() else {}
    facts = load(facts_p) if facts_p.exists() else {}
    out = {}
    for r in records:
        cid, m = r["case_id"], {}
        for prefix, fn in (("so_", so_text), ("subj_", lambda x: (x.get("subjective") or "").strip())):
            s = scorer.score(fn(r["reference_so"]), fn(r[stage]))
            m[prefix + "rouge1_f"] = s["rouge1"].fmeasure
            m[prefix + "rouge2_f"] = s["rouge2"].fmeasure
            m[prefix + "rougeLsum_f"] = s["rougeLsum"].fmeasure
        rows = gcache.get(f"{cid}|{system_key}|v2")
        if rows:
            labels = ["supported" if x["entailment"] >= TE else "contradicted" if x["contradiction"] >= TCONTRA
                      else "uncertain" if x["entailment"] >= TA else "unsupported" for x in rows]
            m["supported_rate"] = labels.count("supported") / len(labels)
            m["unsupported_claim_rate"] = (labels.count("unsupported") + labels.count("contradicted")) / len(labels)
            m["num_claims"] = len(labels)
        fc, cf = fcache.get(cid), facts.get(cid, [])
        if fc and cf and system_key in fc.get("represented", {}):
            rep = fc["represented"][system_key]
            m["atomic_fact_recall"] = sum(rep) / len(cf)
            crit = [i for i, x in enumerate(cf) if x["critical"]]
            if crit:
                m["critical_fact_recall"] = sum(rep[i] for i in crit) / len(crit)
            rec = [i for i in range(len(cf)) if fc["recoverable"][i]]
            if rec:
                m["recall_of_recoverable_facts"] = sum(rep[i] for i in rec) / len(rec)
        out[cid] = m
    return out


def bootstrap_ci(diffs, n=10000, seed=42):
    rnd = random.Random(seed)
    k = len(diffs)
    means = sorted(mean(rnd.choices(diffs, k=k)) for _ in range(n))
    return means[int(0.025 * n)], means[int(0.975 * n) - 1]


def wilcoxon_p(diffs):
    try:
        from scipy.stats import wilcoxon
        if all(abs(d) < 1e-12 for d in diffs):
            return 1.0
        return float(wilcoxon(diffs, zero_method="wilcox").pvalue)
    except Exception as exc:
        return f"n/a ({type(exc).__name__})"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--split", choices=["dev", "test", "all"], default="test")
    ap.add_argument("--umedsum-json", default=str(HERE / "output" / "umedsum_output.json"))
    ap.add_argument("--umedsum-eval-dir", default=str(HERE / "output" / "evaluation"))
    ap.add_argument("--ours-json", default=str(REPO / "verify_and_ask_pipeline" / "Stage23_pipeline" / "output" / "stage23_output.json"))
    ap.add_argument("--ours-eval-dir", default=str(REPO / "verify_and_ask_pipeline" / "Stage23_pipeline" / "output" / "evaluation"))
    ap.add_argument("--out-dir", default=str(HERE / "output" / "comparison"))
    args = ap.parse_args()

    um_eval, ours_eval, out_dir = Path(args.umedsum_eval_dir), Path(args.ours_eval_dir), Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    um_res = load(um_eval / f"results_{args.split}.json")
    ours_res = load(ours_eval / f"results_{args.split}.json")

    # ---------------- aggregate table ----------------
    cols = [("uMedSum S1 (gen)", um_res, "A_stage1"), ("uMedSum S2 (RTB-TS)", um_res, "B_verified"),
            ("uMedSum S3 final", um_res, "C_final"), ("Ours A (Stage 1)", ours_res, "A_stage1"),
            ("Ours B (verified)", ours_res, "B_verified"), ("Ours C final", ours_res, "C_final")]
    metric_names = []
    for _, res, s in cols:
        for k in res["per_system"].get(s, {}):
            if k not in metric_names:
                metric_names.append(k)
    lines = [f"# uMedSum vs Verify-and-Ask - {args.split} split",
             f"uMedSum cases: {um_res['num_cases']} | Verify-and-Ask cases: {ours_res['num_cases']} | "
             f"eval NLI: {um_res.get('eval_nli_model')} / {ours_res.get('eval_nli_model')}",
             "S/O only; A/P is passed through unchanged in both systems and is not scored.", "",
             "| metric | " + " | ".join(c[0] for c in cols) + " | better |",
             "|---|" + "---|" * (len(cols) + 1)]
    for k in metric_names:
        vals = [res["per_system"].get(s, {}).get(k, "") for _, res, s in cols]
        d = DIRECTION.get(k, 1)
        um_v, our_v = vals[2], vals[5]
        better = ""
        if d and isinstance(um_v, (int, float)) and isinstance(our_v, (int, float)) and um_v != our_v:
            better = "ours" if (our_v - um_v) * d > 0 else "uMedSum"
        lines.append(f"| {k} | " + " | ".join(str(v) for v in vals) + f" | {better} |")
    lines += ["", "## Task / integrity metrics (final systems)", "", "| metric | uMedSum | Ours |", "|---|---|---|"]
    tkeys = list(dict.fromkeys(list(um_res["task_metrics"]) + list(ours_res["task_metrics"])))
    for k in tkeys:
        lines.append(f"| {k} | {um_res['task_metrics'].get(k, '')} | {ours_res['task_metrics'].get(k, '')} |")

    # ---------------- paired tests on final notes ----------------
    def recs(p):
        return {r["case_id"]: r for r in load(p)["records"]
                if r.get("status") == "success" and (args.split == "all" or r.get("split") == args.split)}
    um_recs, our_recs = recs(args.umedsum_json), recs(args.ours_json)
    common = sorted(set(um_recs) & set(our_recs))
    test_rows = []
    if common:
        um_pc = per_case([um_recs[c] for c in common], um_eval, "C_final")
        our_pc = per_case([our_recs[c] for c in common], ours_eval, "C_final")
        keys = ["subj_rouge1_f", "subj_rouge2_f", "subj_rougeLsum_f", "so_rouge1_f", "so_rouge2_f", "so_rougeLsum_f",
                "supported_rate", "unsupported_claim_rate", "atomic_fact_recall", "critical_fact_recall",
                "recall_of_recoverable_facts"]
        for k in keys:
            pairs = [(um_pc[c][k], our_pc[c][k]) for c in common if k in um_pc[c] and k in our_pc[c]]
            if len(pairs) < 5:
                continue
            diffs = [o - u for u, o in pairs]
            lo, hi = bootstrap_ci(diffs)
            d = DIRECTION.get(k, 1)
            test_rows.append({"metric": k, "n_cases": len(pairs),
                              "uMedSum_mean": round(mean(u for u, _ in pairs), 4),
                              "ours_mean": round(mean(o for _, o in pairs), 4),
                              "diff_ours_minus_umedsum": round(mean(diffs), 4),
                              "ci95_low": round(lo, 4), "ci95_high": round(hi, 4),
                              "wilcoxon_p": wilcoxon_p(diffs),
                              "ours_wins": sum(1 for x in diffs if x * d > 0),
                              "ties": sum(1 for x in diffs if x == 0),
                              "umedsum_wins": sum(1 for x in diffs if x * d < 0),
                              "higher_is_better": d > 0})
        with open(out_dir / f"paired_tests_{args.split}.csv", "w", newline="", encoding="utf-8") as fh:
            w = csv.DictWriter(fh, fieldnames=list(test_rows[0].keys()) if test_rows else ["metric"])
            w.writeheader()
            w.writerows(test_rows)
        lines += ["", f"## Paired comparison of final notes ({len(common)} shared cases)", "",
                  "| metric | uMedSum | ours | diff (ours-uMedSum) | 95% CI | Wilcoxon p | ours/tie/uMedSum wins |",
                  "|---|---|---|---|---|---|---|"]
        for t in test_rows:
            p = t["wilcoxon_p"]
            p = f"{p:.2g}" if isinstance(p, float) else p
            lines.append(f"| {t['metric']} | {t['uMedSum_mean']} | {t['ours_mean']} | {t['diff_ours_minus_umedsum']:+} | "
                         f"[{t['ci95_low']:+}, {t['ci95_high']:+}] | {p} | {t['ours_wins']}/{t['ties']}/{t['umedsum_wins']} |")
        lines += ["", "Grounding and fact-recall per-case values come from each system's own evaluate.py caches. "
                  "For a strictly matched comparison, both systems should be scored in the same environment "
                  "(see the optional re-scoring cell in the notebook)."]
    else:
        lines += ["", f"No shared successful cases in '{args.split}' between the two output files, so no paired tests. "
                  "(The repo's Verify-and-Ask stage23_output.json holds the 88 test cases only.)"]

    md = "\n".join(lines) + "\n"
    (out_dir / f"comparison_{args.split}.md").write_text(md, encoding="utf-8")
    (out_dir / f"comparison_{args.split}.json").write_text(
        json.dumps({"split": args.split, "umedsum": um_res, "ours": ours_res, "paired_tests": test_rows,
                    "shared_cases": common}, indent=2), encoding="utf-8")
    print(md)
    print("Saved to", out_dir)


if __name__ == "__main__":
    main()

Writing /content/drive/MyDrive/tokenizers/Baseline_Implementation/umedsum_soap_notes/compare_results.py


## 3. Install dependencies, start Ollama, pull LLaMA-3-8B
Verify-and-Ask used LLaMA-3-8B-Instruct **Q8**, so the same quantisation is used here (`llama3:8b-instruct-q8_0`). To match the earlier 140-case baseline run instead, set `MODEL = "llama3:8b"`.

In [4]:
MODEL = "llama3:8b-instruct-q8_0"

!sudo apt-get update -qq && sudo apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
!pip install -q torch transformers accelerate sentencepiece protobuf sentence-transformers keybert pysbd numpy requests rouge-score bert-score scipy

import subprocess, time, os
os.environ["OLLAMA_KEEP_ALIVE"] = "30m"
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama pull {MODEL}
!ollama list

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to GPU")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.

## 4. Smoke test (2 cases)
Check that the S/O/A/P parse says `main_headings` and Stage 2/3 produce sensible counts. The smoke cases are checkpointed under `output/cases/`; they are reused by the full run.

In [5]:
%cd {PIPE}
!python run_umedsum_soap.py --limit 2 --model {MODEL}

/content/drive/MyDrive/tokenizers/Baseline_Implementation/umedsum_soap_notes
Dataset: 110 cases (22 dev / 88 test), seed 42
ICL demonstration(s): ['D2N002'] - none in the test split. Dev cases that are also demos: ['D2N002']
modules.json: 100% 349/349 [00:00<00:00, 1.67MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 718kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 28.5MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 251kB/s]
config.json: 100% 612/612 [00:00<00:00, 3.88MB/s]

model.safetensors: downloading bytes:  94% 85.0M/90.9M [00:00<00:00, 178MB/s, 6.08MB/s  ]
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:00<00:00, 131MB/s, 8.33MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:00<00:00, 140MB/s, 8.94MB/s  ]
Loading weights: 100% 103/103 [00:00<00:00, 5621.22it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.59MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 9.64MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 3

In [6]:
import glob
for p in sorted(glob.glob(f"{OUT}/final_notes/*.md"))[:1]:
    print(open(p).read())
c = json.load(open(sorted(glob.glob(f"{OUT}/cases/*.json"))[0]))
print("parse:", c["stage1_parse"], "\nStage 2:", c["stage2_stats"], "\nStage 3 added:", [a["statement"] for a in c["stage3_added"]])

# D2N001 - uMedSum final note

## Subjective
The patient presents with a history of congestive heart failure, diagnosed at an unknown date.
He reports no major symptoms such as chest pain or shortness of breath.
He is taking medication for heart failure, but sometimes forgets to take his blood pressure medications.
He has not experienced increased shortness of breath.
Patient: I'm 50 years old, and I'm here for my annual exam.
Note that I did not infer or add any information, and I replaced the pronoun "He" with the subject "He" to make each fact self-contained.
Please provide the statement you'd like me to decompose into atomic facts. I'll follow the rules and output the resulting atomic facts.
Splitting the list of symptoms into separate facts
Replacing the pronoun "such as" with the subject "He"
Not adding or inferring any information that is not present in the statement
Outputting each fact as a self-contained proposition
His blood pressure tends to increase during stressful period

## 5. Full run: all 110 cases (22 dev + 88 test)
Resumable: if Colab disconnects, re-run cells 1, 3 and this one, and it continues from the last finished case. Add `--retry-errors` to redo failed cases.

In [7]:
%cd {PIPE}
!python run_umedsum_soap.py --model {MODEL}

/content/drive/MyDrive/tokenizers/Baseline_Implementation/umedsum_soap_notes
Dataset: 110 cases (22 dev / 88 test), seed 42
ICL demonstration(s): ['D2N002'] - none in the test split. Dev cases that are also demos: ['D2N002']
Loading weights: 100% 103/103 [00:00<00:00, 6611.37it/s]
Loading NLI model 'MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli' on cuda...
Loading weights: 100% 394/394 [00:00<00:00, 6004.10it/s]
NLI model loaded successfully! [Entailment: 0, Neutral: 1, Contradiction: 2]
Loading fluency LM 'gpt2' on cuda...
Loading weights: 100% 148/148 [00:00<00:00, 6178.35it/s]
[1/110] D2N001: cached (success)
[2/110] D2N002: cached (success)
[3/110] D2N003 (test) ...[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1366 > 512). Running this sequence through the model will result in indexing errors
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalL

## 6. Evaluate with the Verify-and-Ask evaluator (same code, same reference facts)
The RQ1 `reference_facts.json` is copied in, so fact recall is scored against **the identical fact list** used for our system. Systems `A_stage1 / B_verified / C_final` in this output = uMedSum Stage 1 / Stage 2 / Stage 3.

In [8]:
import shutil
os.makedirs(f"{OUT}/evaluation", exist_ok=True)
shutil.copy(f"{RQ1_EVAL}/reference_facts.json", f"{OUT}/evaluation/reference_facts.json")

# Local models in Colab (no remote servers); evaluator writes to {OUT}/evaluation
os.environ.update({"RQ1_OUTPUT_DIR": OUT, "LLM_BASE_URL": "", "NLI_BASE_URL": "", "EMBED_BASE_URL": "",
                   "OLLAMA_MODEL": MODEL, "PYTHONIOENCODING": "utf-8"})
%cd {RQ1_CODE}
!python evaluate.py --stage23-json {OUT}/umedsum_output.json --split test

/content/drive/MyDrive/tokenizers/verify_and_ask_pipeline/Stage23_pipeline/code
Phase 1/3: ROUGE / BERTScore on 88 cases (loads roberta-large on CPU; a few minutes)
config.json: 100% 482/482 [00:00<00:00, 2.26MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 156kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 21.8MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 26.2MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 14.4MB/s]

model.safetensors: downloading bytes:  30% 432M/1.42G [00:01<00:02, 405MB/s, 35.6MB/s  ]
model.safetensors: downloading bytes:  43% 608M/1.42G [00:02<00:01, 502MB/s, 50.5MB/s  ]
model.safetensors: downloading bytes:  64% 906M/1.42G [00:02<00:00, 548MB/s, 78.7MB/s  ]
model.safetensors: reconstructing file:  66% 938M/1.42G [00:02<00:01, 457MB/s, 31.1MB/s  ]
model.safetensors: reconstructing file:  90% 1.27G/1.42G [00:02<00:00, 631MB/s, 85.6MB/s  ]
model.safetensors: downloading bytes: 100% 952M/952M [00:03<00:00, 268MB/s, 82.3MB/s  ]
model.safetensors: reconst

In [9]:
%cd {RQ1_CODE}
!python evaluate.py --stage23-json {OUT}/umedsum_output.json --split dev

/content/drive/MyDrive/tokenizers/verify_and_ask_pipeline/Stage23_pipeline/code
Phase 1/3: ROUGE / BERTScore on 22 cases (loads roberta-large on CPU; a few minutes)
Loading weights: 100% 389/389 [00:00<00:00, 6883.39it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100% 389/389 [00:00<00:00, 5345.95it/s]
[transformers] RobertaModel LOAD RE

## 7. (Recommended for the final paper numbers) Re-score Verify-and-Ask in this same Colab environment
The saved RQ1 scores were produced on the Windows/GPU-server setup. Re-scoring our outputs here with the same local NLI model and Ollama build removes any environment difference. Output goes to a separate folder; nothing in `Stage23_pipeline/output` is changed.

In [ ]:
RQ1_RESCORED = f"{PIPE}/rq1_rescored"
os.makedirs(f"{RQ1_RESCORED}/evaluation", exist_ok=True)
shutil.copy(f"{ROOT}/verify_and_ask_pipeline/Stage23_pipeline/output/stage23_output.json", f"{RQ1_RESCORED}/stage23_output.json")
shutil.copy(f"{RQ1_EVAL}/reference_facts.json", f"{RQ1_RESCORED}/evaluation/reference_facts.json")
os.environ["RQ1_OUTPUT_DIR"] = RQ1_RESCORED
%cd {RQ1_CODE}
!python evaluate.py --stage23-json {RQ1_RESCORED}/stage23_output.json --split test
os.environ["RQ1_OUTPUT_DIR"] = OUT

## 8. Compare: uMedSum vs Verify-and-Ask
Aggregate table for every metric, plus paired per-case tests on the final notes (mean difference, 95% bootstrap CI, Wilcoxon p, win/tie/loss counts).
Set `OURS_EVAL` to the re-scored folder if you ran step 7, otherwise it uses the saved RQ1 scores.

In [10]:
RESCORED_EVAL = f"{PIPE}/rq1_rescored/evaluation"
OURS_EVAL = RESCORED_EVAL if os.path.exists(f"{RESCORED_EVAL}/results_test.json") else RQ1_EVAL
print("Verify-and-Ask scores from:", OURS_EVAL)
%cd {PIPE}
!python compare_results.py --split test --ours-eval-dir {OURS_EVAL}

Verify-and-Ask scores from: /content/drive/MyDrive/tokenizers/verify_and_ask_pipeline/Stage23_pipeline/output/evaluation
/content/drive/MyDrive/tokenizers/Baseline_Implementation/umedsum_soap_notes
# uMedSum vs Verify-and-Ask - test split
uMedSum cases: 88 | Verify-and-Ask cases: 88 | eval NLI: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli / MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
S/O only; A/P is passed through unchanged in both systems and is not scored.

| metric | uMedSum S1 (gen) | uMedSum S2 (RTB-TS) | uMedSum S3 final | Ours A (Stage 1) | Ours B (verified) | Ours C final | better |
|---|---|---|---|---|---|---|---|
| so_rouge1_f | 0.4549 | 0.4175 | 0.4197 | 0.4533 | 0.4545 | 0.5044 | ours |
| so_rouge2_f | 0.2346 | 0.1704 | 0.1705 | 0.2124 | 0.2132 | 0.2327 | ours |
| so_rougeL_f | 0.2941 | 0.2387 | 0.2371 | 0.2928 | 0.2927 | 0.2773 | ours |
| so_rougeLsum_f | 0.3909 | 0.3607 | 0.3622 | 0.398 | 0.3996 | 0.4377 | ours |
| so_bertscore_f1 | 0.8732 | 0.8

In [11]:
# Dev split: aggregate table only (the repo's stage23_output.json holds the 88 test cases, so no paired tests).
# Note: D2N002 is uMedSum's 1-shot demonstration and is one of the 22 dev cases.
%cd {PIPE}
!python compare_results.py --split dev --ours-eval-dir {RQ1_EVAL}

/content/drive/MyDrive/tokenizers/Baseline_Implementation/umedsum_soap_notes
# uMedSum vs Verify-and-Ask - dev split
uMedSum cases: 22 | Verify-and-Ask cases: 22 | eval NLI: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli / MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
S/O only; A/P is passed through unchanged in both systems and is not scored.

| metric | uMedSum S1 (gen) | uMedSum S2 (RTB-TS) | uMedSum S3 final | Ours A (Stage 1) | Ours B (verified) | Ours C final | better |
|---|---|---|---|---|---|---|---|
| so_rouge1_f | 0.4263 | 0.4172 | 0.422 | 0.4257 | 0.4237 | 0.489 | ours |
| so_rouge2_f | 0.2147 | 0.1615 | 0.1641 | 0.1821 | 0.1796 | 0.2095 | ours |
| so_rougeL_f | 0.2763 | 0.2324 | 0.2345 | 0.2781 | 0.2749 | 0.2624 | ours |
| so_rougeLsum_f | 0.3593 | 0.3502 | 0.3531 | 0.373 | 0.3703 | 0.4155 | ours |
| so_bertscore_f1 | 0.8667 | 0.8516 | 0.8523 | 0.8728 | 0.8722 | 0.8749 | ours |
| subj_rouge1_f | 0.4297 | 0.4201 | 0.4265 | 0.4453 | 0.4421 | 0.4922 | our

## Outputs (all in Drive under `Baseline_Implementation/umedsum_soap_notes/output/`)
| Path | Content |
|---|---|
| `cases/<case>.json` | per-case checkpoint: raw Stage 1 note, S/O/A/P split, Stage 2 audit, Stage 3 insertions |
| `final_notes/<case>.md` | final uMedSum S/O + unchanged A/P |
| `umedsum_output.json` | all cases in the Verify-and-Ask record format (evaluator input) |
| `evaluation/results_{test,dev}.json/.md` | scores (same metrics as ours) |
| `comparison/comparison_{test,dev}.md` | side-by-side table |
| `comparison/paired_tests_test.csv` | per-metric paired tests on the 88 test cases |

**Optional ablation:** `!python run_umedsum_soap.py --model {MODEL} --reproduce-header-bug --out-dir {PIPE}/output_original_filter` reruns uMedSum with the baseline's original Stage 2 line filter.